In [ ]:
import torch
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

weight = 0.7
bias = 0.3

start = 0
end = 1
step = 0.02
x = torch.arange(start, end, step).unsqueeze(dim=1)
y = weight * x + bias

x[:10], y[:10]

In [ ]:
train_split = int(0.8 * len(x))
X_train, y_train = x[:train_split], y[:train_split]
X_test, y_test = x[train_split:], y[train_split:]

len(X_train), len(y_train), len(X_test), len(y_test)

In [ ]:
def plot_pred(train_data=X_train, train_labels=y_train, test_data=X_test, test_labels=y_test, predictions=None):
  plt.figure(figsize=(10,7))
  plt.scatter(train_data, train_labels, c='b', s=4, label='training data')
  plt.scatter(test_data, test_labels, c='g', s=4, label='testing data')
  if predictions is not None:
    plt.scatter(test_data, predictions, c='r', s=4, label='predictions')
  plt.legend(prop={"size": 14})

In [ ]:
plot_pred()

In [ ]:
class LinearRegressionModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(1, dtype=torch.float), requires_grad=True)
    self.bias = nn.Parameter(torch.randn(1, dtype=torch.float), requires_grad=True)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.weights * x + self.bias


In [ ]:
torch.manual_seed(42)

model0 = LinearRegressionModel()
list(model0.parameters())

In [ ]:
model0.state_dict()

In [ ]:
with torch.inference_mode():
  y_preds = model0(X_test)

In [ ]:
print(f"Predicated values:\n{y_preds}",)

In [ ]:
plot_pred(predictions=y_preds)

In [ ]:
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model0.parameters(), lr=0.01)

In [ ]:
torch.manual_seed(42)

epochs = 180

train_loss_values = []
test_loss_values = []
epoch_count = []

for epoch in range(epochs):
  model0.train()
  y_pred = model0(X_train)
  loss = loss_fn(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  model0.eval()
  with torch.inference_mode():
    test_pred = model0(X_test)
    test_loss = loss_fn(test_pred, y_test.type(torch.float))

    if epoch % 10 == 0 :
      epoch_count.append(epoch)
      train_loss_values.append(loss.detach().numpy())
      test_loss_values.append(test_loss.detach().numpy())

      print(f"Epoch: {epoch} | MAE Train Loss: {loss} | MAE Test Loss: {test_loss}")

In [ ]:
plt.plot(epoch_count, train_loss_values,label='train_loss')
plt.plot(epoch_count, test_loss_values, label='test_loss')
plt.title('Training and test loss curves')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()

In [ ]:
print("The model's state_dict:")
print(model0.state_dict())
print("\nAnd the original values for weight and bias:")
print(f"weight: {weight}, bias: {bias}")

In [ ]:
model0.eval()
with torch.inference_mode():
  y_pred_new = model0(X_test)

In [ ]:
plot_pred(predictions=y_preds)

In [ ]:
plot_pred(predictions=y_pred_new)

In [ ]:
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "01_pytorch_workflow.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model0.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
!ls -l models/01_pytorch_workflow.pth

In [ ]:
loaded_model = LinearRegressionModel()
loaded_model.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

In [ ]:
loaded_model.eval()

with torch.inference_mode():
  loaded_model_preds = loaded_model(X_test)

In [ ]:
model0.eval()
with torch.inference_mode():
  y_preds = model0(X_test)

In [ ]:
y_preds == loaded_model_preds